In [1]:
import os
import time
from typing import Optional
import nemo_run as run

devices = 4
# exp_time = "00:15:00" #dev
exp_time = "16:00:00" #dev
# exp_time = "02:00:00" #dev

# TRAINING_SCRIPT = "preprocess_c4_save.py"
# TRAINING_SCRIPT = "model_utils.py"

TRAINING_SCRIPT = "train_def_1024_c4.py"
# TRAINING_SCRIPT = "train_def_768_c4_load.py"

# TRAINING_SCRIPT = "train_minitron.py"
# devices = 1
HOST = "helios" # SWITCH writer entropy helios 

def slurm_executor(
    user: str,
    host: str,
    identity_file: str,
    remote_job_dir: str,
    account: str,
    partition: str,
    nodes: int,
    devices: int,
    time: str = "16:00:00",
    custom_env_vars: Optional[dict[str, str]] = None,
    retries: int = 0,
) -> run.SlurmExecutor:
    # if not (user and host and remote_job_dir and account and partition and nodes and devices):
    #     raise RuntimeError(
    #         "Please set user, host, remote_job_dir, account, partition, nodes, and devices args for using this function."
    #     )

    # Env vars for jobs are configured here
    env_vars = {
        "TORCH_NCCL_AVOID_RECORD_STREAMS": "1",
        "NCCL_NVLS_ENABLE": "0",
        "NVTE_DP_AMAX_REDUCE_INTERVAL": "0",
        "NVTE_ASYNC_AMAX_REDUCTION": "1",
    }
    if custom_env_vars:
        env_vars |= custom_env_vars

    # This will package the train.py script in the current working directory to the remote cluster.
    # If you are inside a git repo, you can also use https://github.com/NVIDIA/NeMo-Run/blob/main/src/nemo_run/core/packaging/git.py.
    # If the script already exists on your container and you call it with the absolute path, you can also just use `run.Packager()`.
    packager = run.PatternPackager(include_pattern=[TRAINING_SCRIPT, "./data/**", "./modelopt/**", "./megatron/**"], relative_path=[os.getcwd(), os.getcwd(), os.getcwd(), os.getcwd()])

    # This defines the slurm executor.
    # We connect to the executor via the tunnel defined by user, host and remote_job_dir.
    executor = run.SlurmExecutor(
        account=account,
        partition=partition,
        tunnel=run.SSHTunnel(
            user=user,
            host=host,
            job_dir=remote_job_dir, # This is where the results of the run will be stored by default.
            identity=identity_file # OPTIONAL: Provide path to the private key that can be used to establish the SSH connection without entering your password.
        ),
        nodes=nodes,
        ntasks_per_node=devices,
        gpus_per_node=devices,
        exclusive=True, #dev SWITHC
        # gres="gpu:4",
        packager=packager,
    )

    executor.env_vars = env_vars
    executor.retries = retries
    executor.time = time
    return executor

if HOST == "helios":
    # Run it locally
    # executor = run.LocalExecutor()
    executor = slurm_executor(
        user="plgmstefaniak",
        host="helios.cyfronet.pl",
        identity_file="/home/maciej/.ssh/plgrid",
        remote_job_dir="/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary",
        # remote_job_dir="net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/llm_random_cemetery",
        # remote_job_dir="/net/home/plgrid/plgmstefaniak/tmp/",
        account="plgllmefficont2-gpu-gh200",
        partition="plgrid-gpu-gh200",
        nodes=1,
        devices=devices, #dev SWITCH 1 4
        time=exp_time

    ) # pass in args relevant to your cluster

    title = "nemo_2_training_experiment"
    exp_id = f"{title}_{int(time.time())}"
    with run.Experiment(title, id=exp_id, log_level="INFO") as exp: #dev bug when you specigy id, expected id = {title}_{id}
        training_job = run.Script(        # --bind /net/scratch/hscra/plgrid/plgmaciejpioro/c4/train:/nemo_run/datasets/c4/train \
        inline=f"""
            python {TRAINING_SCRIPT}
            """,
            entrypoint = f"""apptainer exec --nv \
            --env HOST_PATH=/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/{title}/{exp_id}/training \
            --bind /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/{title}/{exp_id}/training:/nemo_run \
            --bind /net:/net \
            --pwd /nemo_run/code \
            /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/images/nemorand_dev.sif \
            bash"""
            # entrypoint = """
            # pwd ."""
        )
        # training_job = run.Script(
        # inline=f"""
        #     python {TRAINING_SCRIPT}
        #     """,
        #     entrypoint = f"""singularity exec --nv \
        #     --pwd /nemo_run/code \
        #     /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/images/nemorand_dev.sif \
        #     bash"""
        # )
        exp.add(training_job, executor=executor, tail_logs=True, name="training")
        # Add more jobs as needed

        # --bind /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744659529/training/code/preprocessing_results:/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744659529/training/code/preprocessing_results \
        # --bind /net:/net \
        # Run the experiment
        exp.run(detach=False)
elif HOST == "writer":
    executor = slurm_executor(
        user="ubuntu",
        host="164.152.24.115",
        identity_file="/home/maciej/.ssh/llm-random",
        remote_job_dir="/home/ubuntu/llm-random-group/nemo-cementary",
        account=None,
        partition="debug",
        nodes=1,
        devices=devices,
        time=exp_time
    ) # pass in args relevant to your cluster

    title = "nemo_2_training_experiment"
    exp_id = f"{title}_{int(time.time())}"
    with run.Experiment(title, id=exp_id, log_level="INFO") as exp: #dev bug when you specigy id, expected id = {title}_{id}
        training_job = run.Script(        # --bind /net/scratch/hscra/plgrid/plgmaciejpioro/c4/train:/nemo_run/datasets/c4/train \
        inline=f"""
            python {TRAINING_SCRIPT}
            """,
            entrypoint = f"""apptainer exec --nv \
            --env HOST_PATH=/home/ubuntu/llm-random-group/nemo-cementary/{title}/{exp_id}/training \
            --bind /home/ubuntu/llm-random-group/nemo-cementary/{title}/{exp_id}/training:/nemo_run \
            --bind /home/ubuntu/llm-random-group/nemo-cementary:/home/ubuntu/llm-random-group/nemo-cementary \
            --pwd /nemo_run/code \
            /home/ubuntu/mstefaniak/nemo/nemorand_dev.sif \
            bash"""
            # entrypoint = """
            # pwd ."""
        )
        exp.add(training_job, executor=executor, tail_logs=True, name="training")
        exp.run(detach=False)
elif HOST == "entropy":
    executor = slurm_executor(
        user="mstefaniak",
        host="entropy.mimuw.edu.pl",
        identity_file="/home/maciej/.ssh/id_rsa",
        remote_job_dir="/home/mstefaniak/nemo_cementary",
        # account="mim",
        account="mim",
        partition="h100",
        nodes=1,
        devices=1, #dev SWITCH 1 4
        time=exp_time
    ) # pass in args relevant to your cluster

    title = "nemo_2_training_experiment"
    exp_id = f"{title}_{int(time.time())}"
    with run.Experiment(title, id=exp_id, log_level="INFO") as exp: #dev bug when you specigy id, expected id = {title}_{id}
        training_job = run.Script(        # --bind /net/scratch/hscra/plgrid/plgmaciejpioro/c4/train:/nemo_run/datasets/c4/train \
        inline=f"""
            python {TRAINING_SCRIPT}
            """,
            entrypoint = f"""singularity exec --nv \
            --env HOST_PATH=/home/mstefaniak/nemo_cementary/{title}/{exp_id}/training \
            --bind /home/mstefaniak/nemo_cementary/{title}/{exp_id}/training:/nemo_run \
            --bind /home/mstefaniak:/home/mstefaniak \
            --pwd /nemo_run/code \
            /home/mstefaniak/mas/nemo/mstest/nemorand_dev.sif \
            bash"""
        )
        exp.add(training_job, executor=executor, tail_logs=True, name="training")
        exp.run(detach=False)

# gpt2 tokenizer: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744832098/training
# gpt2 megatron - provided files: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744842516/training - /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744842516/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document
# gpt2 hf (daa) auto: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744842905/training - /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744842905/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document
# gpt2 hf megatron auto cached files: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744843405/training - /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744843405/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document

# training gpt2 auto fix: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744878751/training - /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744878751/training/code/checkpoints/nemotron/default/2025-04-17_10-33-24/checkpoints/default--None=0.0000-epoch=0-consumed_samples=7680000.0/weights

# latest: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744891487/training
# dev 100 with checkpoint: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744931414/training
# 
# model save: 
# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1745317560/training
# x4 training: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1745363165/training
# 
# Prunned compute optimal old hparams: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1745586856/training/code/prrruned_nyan
# 
# 
# 
 



────────── Entering Experiment nemo_2_training_experiment with id: nemo_2_training_experiment_1747055740 ──────────

[15:15:41] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=422821;file:///home/maciej/code/llm-random/where_nemo/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=74277;file:///home/maciej/code/llm-random/where_nemo/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!
rsyncing /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1747055740 to /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment ...
Successfully ran `rsync  -pthrvz  --rsh='ssh -i /home/maciej/.ssh/plgrid -p 22 ' /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1747055740 plgmstefaniak@helios.cyfronet.pl:/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment`


[15:15:49] Launching job training for experiment nemo_2_training_experiment                       ]8;id=822579;file:///home/maciej/code/llm-random/where_nemo/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=530896;file:///home/maciej/code/llm-random/where_nemo/nemo_run/run/experiment.py#744\744]8;;\

Launched app: slurm_tunnel://nemo_run/634118


...

Waiting for job 634118 to finish [log=True]...
[chan 12] Opened sftp connection (server version 3)


training/0 [NeMo W 2025-05-12 15:16:39 nemo_logging:405] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
training/0 [rank: 0] Seed set to 27
training/0 [rank: 2] Seed set to 27
training/0 [rank: 3] Seed set to 27
training/0 [rank: 1] Seed set to 27
training/0 Its TRAINING BS ---------------------------------------------------------------
training/0 ['/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1746522127/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document']
training/0 Its TRAINING BS ---------------------------------------------------------------
training/0 ['/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1746522127/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document']
training/0 Its TRAINING BS 

Job 634118 finished: SUCCEEDED


                                                                                                                   
# The experiment was run with the following tasks: ['training']                                                    
# You can inspect and reconstruct this experiment at a later point in time using:                                  
experiment = run.Experiment.from_id("nemo_2_training_experiment_1747055740")                                       
experiment.status() # Gets the overall status                                                                      
experiment.logs("training") # Gets the log for the provided task                                                   
experiment.cancel("training") # Cancels the provided task if still running                                         
                                                                                                                   

                                                                                                                   
# You can inspect this experiment at a later point in time using the CLI as well:                                  
nemo experiment status nemo_2_training_experiment_1747055740                                                       
nemo experiment logs nemo_2_training_experiment_1747055740 0                                                       
nemo experiment cancel nemo_2_training_experiment_1747055740 0                                                     
                                                                                                                   

In [2]:
# (base) [helios][plgmstefaniak@login01 results_preprocessing]$ ls
# c4_en_train_part_00.jsonl_text_document      c4_en_train_part_01.jsonl_text_document.bin  c4_en_train_part_02.jsonl_text_document.idx  c4_en_train_part_04.jsonl_text_document.bin  c4_en_train_part_05.jsonl_text_document.idx  c4_en_train_part_07.jsonl_text_document.bin  c4_en_train_part_08.jsonl_text_document.idx
# c4_en_train_part_00.jsonl_text_document.bin  c4_en_train_part_01.jsonl_text_document.idx  c4_en_train_part_03.jsonl_text_document.bin  c4_en_train_part_04.jsonl_text_document.idx  c4_en_train_part_06.jsonl_text_document.bin  c4_en_train_part_07.jsonl_text_document.idx  c4_en_train_part_09.jsonl_text_document.bin
# c4_en_train_part_00.jsonl_text_document.idx  c4_en_train_part_02.jsonl_text_document.bin  c4_en_train_part_03.jsonl_text_document.idx  c4_en_train_part_05.jsonl_text_document.bin  c4_en_train_part_06.jsonl_text_document.idx  c4_en_train_part_08.jsonl_text_document.bin  c4_en_train_part_09.jsonl_text_document.idx
# (base) [helios][plgmstefaniak@login01 results_preprocessing]$ pwd
# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1744842905/training/code/results_preprocessing


# # srun 
# --output /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training/log-plgllmefficont2-gpu-gh200-plgllmefficont2-gpu-gh200.training_%j_${SLURM_RESTART_COUNT:-0}.out 
# --container-mounts /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/nemo_2.0_training_experiment_1744148528/training:/nemo_run 
# --container-workdir /nemo_run/code 
# --wait=60 --kill-on-bad-exit=1 bash /nemo_run/scripts/training.sh


In [3]:
import nemo_run as run

experiment = run.Experiment.from_id(exp_id)                                 
experiment.status() # Gets the overall status                                                                      
experiment.logs("training") # Gets the log for the provided task                                                   
# experiment.cancel("training") # Cancels the provided task if still running

# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/1744234741/training
# /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2.0_training_experiment/1744234741/training

[16:25:38] Connecting to plgmstefaniak@helios.cyfronet.pl                                             ]8;id=717027;file:///home/maciej/code/llm-random/where_nemo/nemo_run/core/tunnel/client.py\client.py]8;;\:]8;id=939049;file:///home/maciej/code/llm-random/where_nemo/nemo_run/core/tunnel/client.py#257\257]8;;\

Connected (version 2.0, client OpenSSH_8.0)
Authentication (publickey) successful!


Experiment Status for nemo_2_training_experiment_1747055740

Task 0: training
- Status: SUCCEEDED
- Executor: SlurmExecutor on plgmstefaniak@helios.cyfronet.pl
- Job id: 634118
- Local Directory: /home/maciej/.nemo_run/experiments/nemo_2_training_experiment/nemo_2_training_experiment_1747055740/training
- Remote Directory: /net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1747055740/training

[16:25:42] Fetching logs for training                                                             ]8;id=948450;file:///home/maciej/code/llm-random/where_nemo/nemo_run/run/experiment.py\experiment.py]8;;\:]8;id=390376;file:///home/maciej/code/llm-random/where_nemo/nemo_run/run/experiment.py#931\931]8;;\

[chan 7] Opened sftp connection (server version 3)
training/0 [NeMo W 2025-05-12 15:16:39 nemo_logging:405] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
training/0 [rank: 0] Seed set to 27
training/0 [rank: 2] Seed set to 27
training/0 [rank: 3] Seed set to 27
training/0 [rank: 1] Seed set to 27
training/0 Its TRAINING BS ---------------------------------------------------------------
training/0 ['/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1746522127/training/code/results_preprocessing/c4_en_train_part_00.jsonl_text_document']
training/0 Its TRAINING BS ---------------------------------------------------------------
training/0 ['/net/storage/pr3/plgrid/plggllmeffi/plgmstefaniak/nemo_cementary/nemo_2_training_experiment/nemo_2_training_experiment_1746522127/training/code/results_preprocessing/c4_en_train_part_0

In [4]:
experiment

Graphviz rendering failed: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH
